# 04 — Baselines and statistical forecasting models

This notebook reproduces the reference and statistical model families used in the greenhouse temperature and relative humidity benchmark.

It reads the five temporal-resolution datasets created by `03_multiresolution_dataset.ipynb`, constructs one common set of forecast origins per resolution, evaluates four reference models, tunes direct autoregressive Ridge models on the validation partition, refits them on training plus validation data, and evaluates the final models on the independent test partition.

All paths are relative to the repository root. Run notebooks `01`, `02`, and `03` before this notebook.


In [ ]:
from pathlib import Path
import json
import platform
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )

PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "baselines_statistical"
MODEL_DIR = PROJECT_ROOT / "models" / "baselines_statistical"
FIGURE_DIR = PROJECT_ROOT / "figures" / "baselines_statistical"
METADATA_DIR = PROJECT_ROOT / "metadata"

for directory in [INDEX_DIR, RESULTS_DIR, MODEL_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")


## Experimental configuration

The common-origin rule follows the original experiment: a 24-hour historical window, no more than 15% interpolated intervals in that window, valid targets at all four forecast horizons, and no forecast crossing a chronological split boundary. Historical bins must have at least 75% target-pair availability.


In [ ]:
RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24
ROLLING_MEAN_HOURS = 6
MINIMUM_HISTORY_AVAILABILITY = 0.75
MAXIMUM_INTERPOLATED_FRACTION = 0.15
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
RIDGE_SOLVER = "lsqr"
REFERENCE_MODELS = [
    "PERSISTENCE",
    "SEASONAL_NAIVE_24H",
    "ROLLING_MEAN_6H",
    "DIURNAL_CLIMATOLOGY",
]
STATISTICAL_MODELS = ["AR_DIRECT_RIDGE", "VAR_DIRECT_RIDGE"]
EVALUATION_SPLITS = ["validation", "test"]

configuration = {
    "temporal_resolutions_minutes": RESOLUTIONS,
    "targets": TARGETS,
    "forecast_horizons_minutes": HORIZONS_MINUTES,
    "history_hours": HISTORY_HOURS,
    "rolling_mean_hours": ROLLING_MEAN_HOURS,
    "minimum_history_availability": MINIMUM_HISTORY_AVAILABILITY,
    "maximum_interpolated_fraction": MAXIMUM_INTERPOLATED_FRACTION,
    "ridge_alpha_grid": RIDGE_ALPHAS,
    "ridge_solver": RIDGE_SOLVER,
    "reference_models": REFERENCE_MODELS,
    "statistical_models": STATISTICAL_MODELS,
    "random_seed": RANDOM_SEED,
    "sample_comparison_policy": "All models use the same effective forecast origins within each resolution.",
    "split_policy": "The origin and all future targets remain inside the same chronological split.",
}
(METADATA_DIR / "04_baselines_statistical_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
configuration


In [ ]:
def load_resolution(resolution_minutes):
    path = RESOLUTION_DIR / f"greenhouse_{resolution_minutes}min.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path.relative_to(PROJECT_ROOT)}. Run notebook 03 first.")
    data = pd.read_csv(path, parse_dates=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)
    if data["target_valid_for_evaluation_flag"].dtype != bool:
        data["target_valid_for_evaluation_flag"] = (
            data["target_valid_for_evaluation_flag"].astype(str).str.lower().eq("true")
        )
    return data


def build_effective_origins(data, resolution_minutes):
    n = len(data)
    history_steps = HISTORY_HOURS * 60 // resolution_minutes
    horizon_steps = {h: h // resolution_minutes for h in HORIZONS_MINUTES}
    maximum_horizon_steps = max(horizon_steps.values())
    positions = np.arange(n)

    enough_context = (positions >= history_steps - 1) & (
        positions + maximum_horizon_steps < n
    )
    history_complete = (
        data["target_pair_available_fraction"]
        .ge(MINIMUM_HISTORY_AVAILABILITY)
        .astype(int)
        .rolling(history_steps, min_periods=history_steps)
        .sum()
        .eq(history_steps)
        .to_numpy()
    )
    interpolation_ok = (
        data["target_pair_interpolated_fraction"]
        .rolling(history_steps, min_periods=history_steps)
        .mean()
        .le(MAXIMUM_INTERPOLATED_FRACTION)
        .fillna(False)
        .to_numpy()
    )

    targets_valid = np.ones(n, dtype=bool)
    targets_same_split = np.ones(n, dtype=bool)
    for steps in horizon_steps.values():
        targets_valid &= (
            data["target_valid_for_evaluation_flag"].shift(-steps).fillna(False).to_numpy(dtype=bool)
        )
        targets_same_split &= (
            data["split"].shift(-steps).eq(data["split"]).fillna(False).to_numpy()
        )

    valid = enough_context & history_complete & interpolation_ok & targets_valid & targets_same_split
    origins = positions[valid]
    result = pd.DataFrame({
        "origin_index": origins,
        "origin_timestamp": data.loc[origins, "timestamp"].to_numpy(),
        "split": data.loc[origins, "split"].to_numpy(),
        "history_start_index": origins - history_steps + 1,
        "history_end_index": origins,
        "history_steps": history_steps,
        "history_hours": HISTORY_HOURS,
        "maximum_interpolated_fraction": MAXIMUM_INTERPOLATED_FRACTION,
    })
    for horizon, steps in horizon_steps.items():
        result[f"target_index_h{horizon}"] = origins + steps
        result[f"target_timestamp_h{horizon}"] = data.loc[origins + steps, "timestamp"].to_numpy()

    audit = {
        "resolution": f"{resolution_minutes}min",
        "resolution_minutes": resolution_minutes,
        "rows": n,
        "enough_context": int(enough_context.sum()),
        "history_complete": int((enough_context & history_complete).sum()),
        "interpolation_ok": int((enough_context & history_complete & interpolation_ok).sum()),
        "targets_valid": int((enough_context & history_complete & interpolation_ok & targets_valid).sum()),
        "targets_same_split": int(valid.sum()),
        "train_samples": int((result["split"] == "train").sum()),
        "validation_samples": int((result["split"] == "validation").sum()),
        "test_samples": int((result["split"] == "test").sum()),
    }
    return result, audit


datasets = {}
effective_origins = {}
audit_rows = []
for resolution in RESOLUTIONS:
    data = load_resolution(resolution)
    origins, audit = build_effective_origins(data, resolution)
    datasets[resolution] = data
    effective_origins[resolution] = origins
    origins.to_csv(INDEX_DIR / f"effective_indices_{resolution}min.csv", index=False)
    audit_rows.append(audit)

sample_audit = pd.DataFrame(audit_rows)
sample_audit.to_csv(RESULTS_DIR / "01_effective_sample_audit.csv", index=False)
display(sample_audit)


In [ ]:
EXPECTED_COUNTS = {
    4: {"train": 1053, "validation": 1446, "test": 2742},
    12: {"train": 104, "validation": 135, "test": 378},
    20: {"train": 146, "validation": 239, "test": 556},
    30: {"train": 104, "validation": 142, "test": 350},
    60: {"train": 66, "validation": 58, "test": 211},
}

count_checks = []
for resolution, origins in effective_origins.items():
    for split, expected in EXPECTED_COUNTS[resolution].items():
        observed = int((origins["split"] == split).sum())
        count_checks.append({
            "resolution": f"{resolution}min",
            "split": split,
            "observed": observed,
            "expected": expected,
            "match": observed == expected,
        })
count_checks = pd.DataFrame(count_checks)
count_checks.to_csv(RESULTS_DIR / "02_historical_count_check.csv", index=False)
display(count_checks)
assert count_checks["match"].all(), "Effective sample counts differ from the audited historical experiment."


## Reference models

The reference family contains persistence, a 24-hour seasonal naive forecast, a six-hour rolling mean, and a diurnal climatology estimated only from data available in the fitting scope. Validation predictions use training data; test predictions use training plus validation data.


In [ ]:
def minute_of_day(timestamps):
    timestamps = pd.DatetimeIndex(timestamps)
    return timestamps.hour * 60 + timestamps.minute


def climatology_lookup(data, target, fitting_splits):
    mask = data["split"].isin(fitting_splits) & data[target].notna()
    values = data.loc[mask, ["timestamp", target]].copy()
    values["minute_of_day"] = minute_of_day(values["timestamp"])
    lookup = values.groupby("minute_of_day")[target].mean()
    fallback = float(values[target].mean())
    return lookup, fallback


baseline_prediction_rows = []
for resolution, data in datasets.items():
    origins = effective_origins[resolution]
    seasonal_steps = 24 * 60 // resolution
    rolling_steps = ROLLING_MEAN_HOURS * 60 // resolution
    for split in EVALUATION_SPLITS:
        fitting_splits = ["train"] if split == "validation" else ["train", "validation"]
        split_origins = origins.loc[origins["split"] == split]
        origin_indices = split_origins["origin_index"].to_numpy(dtype=int)
        for target in TARGETS:
            target_values = data[target].to_numpy(dtype=float)
            lookup, fallback = climatology_lookup(data, target, fitting_splits)
            for horizon in HORIZONS_MINUTES:
                horizon_steps = horizon // resolution
                target_indices = origin_indices + horizon_steps
                target_timestamps = data.loc[target_indices, "timestamp"].to_numpy()
                observed = target_values[target_indices]
                predictions = {
                    "PERSISTENCE": target_values[origin_indices],
                    "SEASONAL_NAIVE_24H": target_values[target_indices - seasonal_steps],
                    "ROLLING_MEAN_6H": np.array([
                        np.mean(target_values[index - rolling_steps + 1:index + 1])
                        for index in origin_indices
                    ]),
                    "DIURNAL_CLIMATOLOGY": np.array([
                        lookup.get(value, fallback) for value in minute_of_day(target_timestamps)
                    ]),
                }
                for model, predicted in predictions.items():
                    baseline_prediction_rows.append(pd.DataFrame({
                        "resolution": f"{resolution}min",
                        "resolution_minutes": resolution,
                        "model": model,
                        "family": "reference",
                        "split": split,
                        "target": target,
                        "horizon_minutes": horizon,
                        "origin_index": origin_indices,
                        "origin_timestamp": split_origins["origin_timestamp"].to_numpy(),
                        "target_timestamp": target_timestamps,
                        "observed": observed,
                        "predicted": predicted,
                    }))

baseline_predictions = pd.concat(baseline_prediction_rows, ignore_index=True)
baseline_predictions.to_csv(RESULTS_DIR / "03_baseline_predictions.csv", index=False)
baseline_predictions.head()


## Direct autoregressive Ridge models

`AR_DIRECT_RIDGE` uses the 24-hour history of the target variable. `VAR_DIRECT_RIDGE` uses the aligned 24-hour histories of both temperature and relative humidity. Each target–horizon model is tuned independently on validation MAE. Predictors are standardized using training-only statistics.


In [ ]:
def build_feature_matrix(data, origin_indices, history_steps, model, target):
    if model == "AR_DIRECT_RIDGE":
        columns = [target]
    elif model == "VAR_DIRECT_RIDGE":
        columns = TARGETS
    else:
        raise ValueError(model)
    values = data[columns].to_numpy(dtype=float)
    return np.stack([
        values[index - history_steps + 1:index + 1].reshape(-1)
        for index in origin_indices
    ])


statistical_prediction_rows = []
tuning_rows = []
training_rows = []

for resolution, data in datasets.items():
    origins = effective_origins[resolution]
    origin_indices = origins["origin_index"].to_numpy(dtype=int)
    split_labels = origins["split"].to_numpy()
    history_steps = HISTORY_HOURS * 60 // resolution
    train_mask = split_labels == "train"
    validation_mask = split_labels == "validation"
    final_fit_mask = np.isin(split_labels, ["train", "validation"])
    test_mask = split_labels == "test"

    for model_name in STATISTICAL_MODELS:
        for target in TARGETS:
            X = build_feature_matrix(data, origin_indices, history_steps, model_name, target)
            target_values = data[target].to_numpy(dtype=float)
            for horizon in HORIZONS_MINUTES:
                horizon_steps = horizon // resolution
                y = target_values[origin_indices + horizon_steps]
                validation_scores = []
                validation_models = []
                for alpha in RIDGE_ALPHAS:
                    candidate = make_pipeline(
                        StandardScaler(),
                        Ridge(alpha=alpha, solver=RIDGE_SOLVER),
                    )
                    started = time.perf_counter()
                    candidate.fit(X[train_mask], y[train_mask])
                    fit_seconds = time.perf_counter() - started
                    started = time.perf_counter()
                    validation_prediction = candidate.predict(X[validation_mask])
                    inference_seconds = time.perf_counter() - started
                    validation_mae = mean_absolute_error(y[validation_mask], validation_prediction)
                    validation_scores.append(validation_mae)
                    validation_models.append(candidate)
                    tuning_rows.append({
                        "resolution": f"{resolution}min",
                        "model": model_name,
                        "target": target,
                        "horizon_minutes": horizon,
                        "alpha": alpha,
                        "validation_mae": validation_mae,
                        "fit_seconds": fit_seconds,
                        "inference_seconds": inference_seconds,
                    })

                best_position = int(np.argmin(validation_scores))
                selected_alpha = RIDGE_ALPHAS[best_position]
                validation_model = validation_models[best_position]
                validation_prediction = validation_model.predict(X[validation_mask])

                final_model = make_pipeline(
                    StandardScaler(),
                    Ridge(alpha=selected_alpha, solver=RIDGE_SOLVER),
                )
                started = time.perf_counter()
                final_model.fit(X[final_fit_mask], y[final_fit_mask])
                final_fit_seconds = time.perf_counter() - started
                started = time.perf_counter()
                test_prediction = final_model.predict(X[test_mask])
                test_inference_seconds = time.perf_counter() - started

                model_path = (
                    MODEL_DIR / f"{resolution}min" / model_name /
                    f"{target}_h{horizon}min.joblib"
                )
                model_path.parent.mkdir(parents=True, exist_ok=True)
                joblib.dump(final_model, model_path)

                training_rows.append({
                    "resolution": f"{resolution}min",
                    "model": model_name,
                    "target": target,
                    "horizon_minutes": horizon,
                    "selected_alpha": selected_alpha,
                    "history_hours": HISTORY_HOURS,
                    "maximum_interpolated_fraction": MAXIMUM_INTERPOLATED_FRACTION,
                    "n_features_flattened": X.shape[1],
                    "train_samples": int(train_mask.sum()),
                    "validation_samples": int(validation_mask.sum()),
                    "test_samples": int(test_mask.sum()),
                    "final_fit_seconds": final_fit_seconds,
                    "test_inference_seconds": test_inference_seconds,
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                })

                for split, mask, predicted in [
                    ("validation", validation_mask, validation_prediction),
                    ("test", test_mask, test_prediction),
                ]:
                    selected_origins = origins.loc[mask]
                    selected_indices = origin_indices[mask]
                    target_indices = selected_indices + horizon_steps
                    statistical_prediction_rows.append(pd.DataFrame({
                        "resolution": f"{resolution}min",
                        "resolution_minutes": resolution,
                        "model": model_name,
                        "family": "statistical",
                        "split": split,
                        "target": target,
                        "horizon_minutes": horizon,
                        "origin_index": selected_indices,
                        "origin_timestamp": selected_origins["origin_timestamp"].to_numpy(),
                        "target_timestamp": data.loc[target_indices, "timestamp"].to_numpy(),
                        "observed": y[mask],
                        "predicted": predicted,
                    }))

statistical_predictions = pd.concat(statistical_prediction_rows, ignore_index=True)
ridge_tuning = pd.DataFrame(tuning_rows)
training_summary = pd.DataFrame(training_rows)
statistical_predictions.to_csv(RESULTS_DIR / "04_statistical_predictions.csv", index=False)
ridge_tuning.to_csv(RESULTS_DIR / "05_ridge_tuning.csv", index=False)
training_summary.to_csv(RESULTS_DIR / "06_model_training_summary.csv", index=False)
display(training_summary.head(12))


## Metrics and rankings

MAE, RMSE, bias, $R^2$, sMAPE, and seasonal MASE are calculated for every resolution–model–target–horizon–split combination. The MASE denominator is estimated only from the training partition using a 24-hour seasonal difference.


In [ ]:
def mase_scale(data, target, resolution):
    training = data.loc[data["split"] == "train", target].to_numpy(dtype=float)
    seasonal_steps = 24 * 60 // resolution
    differences = np.abs(training[seasonal_steps:] - training[:-seasonal_steps])
    differences = differences[np.isfinite(differences)]
    return float(np.mean(differences))


mase_scales = {
    (resolution, target): mase_scale(data, target, resolution)
    for resolution, data in datasets.items()
    for target in TARGETS
}


def metric_row(group):
    observed = group["observed"].to_numpy(dtype=float)
    predicted = group["predicted"].to_numpy(dtype=float)
    error = predicted - observed
    denominator = np.abs(observed) + np.abs(predicted)
    smape = 200 * np.mean(np.divide(
        np.abs(error), denominator,
        out=np.zeros_like(error), where=denominator > 1e-12,
    ))
    resolution = int(group.name[1])
    target = group.name[5]
    return pd.Series({
        "n": len(group),
        "mae": mean_absolute_error(observed, predicted),
        "rmse": mean_squared_error(observed, predicted) ** 0.5,
        "bias": float(np.mean(error)),
        "r2": r2_score(observed, predicted),
        "smape_pct": smape,
        "mase": mean_absolute_error(observed, predicted) / mase_scales[(resolution, target)],
    })


all_predictions = pd.concat([baseline_predictions, statistical_predictions], ignore_index=True)
group_columns = [
    "resolution", "resolution_minutes", "model", "family",
    "split", "target", "horizon_minutes",
]
all_metrics = (
    all_predictions.groupby(group_columns, observed=True, sort=False)
    .apply(metric_row, include_groups=False)
    .reset_index()
)
all_predictions.to_csv(RESULTS_DIR / "07_all_predictions.csv", index=False)
all_metrics.to_csv(RESULTS_DIR / "08_all_metrics.csv", index=False)

validation_ranking = all_metrics.loc[all_metrics["split"] == "validation"].copy()
validation_ranking["mae_rank"] = validation_ranking.groupby(
    ["resolution", "target", "horizon_minutes"]
)["mae"].rank(method="min")
test_ranking = all_metrics.loc[all_metrics["split"] == "test"].copy()
test_ranking["mae_rank"] = test_ranking.groupby(
    ["resolution", "target", "horizon_minutes"]
)["mae"].rank(method="min")
validation_ranking.to_csv(RESULTS_DIR / "09_validation_ranking.csv", index=False)
test_ranking.to_csv(RESULTS_DIR / "10_test_ranking.csv", index=False)
display(test_ranking.sort_values(["target", "resolution_minutes", "horizon_minutes", "mae_rank"]).head(20))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for axis, target in zip(axes, TARGETS):
    subset = all_metrics.loc[
        (all_metrics["split"] == "test") & (all_metrics["target"] == target)
    ]
    sns.lineplot(
        data=subset, x="horizon_minutes", y="mae",
        hue="model", style="resolution", markers=True, dashes=False, ax=axis,
    )
    axis.set_title(target.replace("_", " ").title())
    axis.set_xlabel("Forecast horizon (min)")
    axis.set_ylabel("Test MAE")
    axis.legend(fontsize=7, ncol=2)
figure_path = FIGURE_DIR / "04_test_mae_by_horizon.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {figure_path.relative_to(PROJECT_ROOT)}")


In [ ]:
output_summary = pd.DataFrame({
    "artifact": [
        "effective sample files", "reference predictions", "statistical predictions",
        "trained Ridge models", "metric rows", "diagnostic figures",
    ],
    "count": [
        len(list(INDEX_DIR.glob("effective_indices_*min.csv"))),
        len(baseline_predictions),
        len(statistical_predictions),
        len(list(MODEL_DIR.rglob("*.joblib"))),
        len(all_metrics),
        len(list(FIGURE_DIR.glob("*.png"))),
    ],
})
display(output_summary)
print(f"Results written to: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models written to: {MODEL_DIR.relative_to(PROJECT_ROOT)}")


## Reproducibility note

The notebook asserts the audited effective-sample counts before fitting any model. A failed assertion indicates that an upstream dataset, split boundary, availability threshold, or interpolation rule has changed and should be investigated before comparing performance results.
